# Browser check

Run every cell, top to bottom, and read the **PASS / FAIL** lines.
Then say which browser and version you were on.

The Lab is tested in Chromium. Firefox and Safari are untested, and
the point of this notebook is to find out what breaks there without
needing a debugger or a checkout — open the page, hit **Run all**,
read the results.

Everything below is either a fact about the running kernel or a fact
about the page. Nothing here is a benchmark; a slow PASS is a PASS.


## 1. The kernel is alive and answering

If this fails, nothing else matters.


In [ ]:
local ok = _VERSION ~= nil
print((ok and "PASS" or "FAIL") .. "  kernel: " .. tostring(_VERSION))


## 2. `pcall` catches — the one that matters most

Lua's `pcall` is `setjmp`/`longjmp`, which the toolchain lowers onto
WebAssembly exception handling. Engines shipped that at different
times. If it is missing, **`pcall` stops catching and every other
test still passes** — a broken language with a green page. This is
the single most important line in the notebook.


In [ ]:
local caught, err = pcall(function() error("boom") end)
local nested = pcall(function() pcall(function() error("inner") end) error("outer") end)
print(((not caught) and "PASS" or "FAIL") .. "  pcall catches a plain error")
print(((err and tostring(err):find("boom")) and "PASS" or "FAIL") .. "  the message survives: " .. tostring(err))
print(((not nested) and "PASS" or "FAIL") .. "  nested pcall unwinds correctly")


## 3. Errors are still catchable after a deep stack

Same machinery, further from the top. `error` thrown 200 frames down
has to unwind all of them.

Note this is a *recursive* call, not a tail call: `1 + deep(n-1)`
rather than `return deep(n-1)`. A tail call reuses the frame, so it
never overflows and never returns — it just hangs.


In [ ]:
local function deep(n) if n == 0 then error("from the bottom") end return 1 + deep(n - 1) end
local ok, err = pcall(deep, 200)
print(((not ok) and "PASS" or "FAIL") .. "  a deep error unwinds: " .. tostring(err))


## 4. Numbers, strings and the standard library


In [ ]:
local checks = {
  { "integers are 64-bit", math.maxinteger == 9223372036854775807 },
  { "integer division",    7 // 2 == 3 },
  { "float division",      7 / 2 == 3.5 },
  { "string.format",       string.format("%.2f", 1.5) == "1.50" },
  { "utf8 library",        utf8 ~= nil and utf8.len("héllo") == 5 },
  { "table.concat",        table.concat({"a","b"}, "-") == "a-b" },
  { "os.time exists",      type(os.time()) == "number" },
}
for _, c in ipairs(checks) do print((c[2] and "PASS" or "FAIL") .. "  " .. c[1]) end


## 5. Output: many small writes, and non-ASCII across them

`print` emits one write per argument. Text is decoded incrementally,
so a multi-byte character split across two writes decodes to a
replacement character if the decoder is not streaming.


In [ ]:
print("PASS  ascii output")
print("check the next line reads: 世界 🌊 café")
print("世界", "🌊", "café")
io.write("check this line ends with three dots") io.write(".") io.write(".") io.write(".") print()


## 6. Output caps

One runaway cell must not take the tab with it. This prints far more
than the cap allows and should be **truncated with a note**, not
dropped and not endless.


In [ ]:
for i = 1, 5000 do print("line " .. i) end
print("PASS  reached the end; look for a truncation note above")


## 7. The value echo

A cell ending in an expression shows its value. Tables show their
contents rather than an address.


In [ ]:
{ name = "diluvium", version = _VERSION, list = { 1, 2, 3 } }


## 8. State persists between cells

Run this cell twice. The number must go up.


In [ ]:
counter = (counter or 0) + 1
print("run count: " .. counter .. "   (run this cell again — it must increase)")


## 9. Compiling without running

Press **Bytecode** on the cell below. It should show a disassembly
and `hello from the bytecode viewer` must **not** appear in the
output — compiling is not running.


In [ ]:
print("hello from the bytecode viewer")


## 10. Stop a runaway cell

**This one is manual, and it is the important one.**

Run the cell below. It never finishes. Then:

1. Check the page still responds — scroll, type in another cell.
2. Press **Stop** in the toolbar.

PASS if the page stayed responsive *and* Stop ended it. FAIL — and
worth reporting loudly — if the whole tab froze. On a browser with no
worker support the Lab runs the kernel in the page and disables Stop;
in that case the tab freezing is expected and the kernel label in the
toolbar will say `in page`.


In [ ]:
-- Runs forever on purpose. Press Stop.
while true do end


## 11. Things about the page rather than the kernel

These are checked by the Lab itself and reported in the console
panel at the bottom of the window. Nothing to run — just look at the
kernel label in the toolbar:

- **`(worker)`** — the kernel is off the main thread and Stop works.
- **`(in page)`** — no worker; a runaway cell will freeze the tab.

Then try, by hand:

1. **Undo.** Type in a cell, press Ctrl+Z (Cmd+Z). The text should
   come back — the editor uses the browser's own undo stack.
2. **Selection and caret.** Select some code. The selection must be
   visible and the caret must be drawn.
3. **Ctrl+/** on a line — it should comment and uncomment.
4. **Tab / Shift+Tab** in a cell — indent and dedent.
5. **Save .ipynb**, then **Open…** it again. The notebook should
   come back unchanged.
6. **Reload the page.** Your cells should still be there.
7. **Runtime dropdown**, press ⟳. It should list builds from the
   mirror, or say plainly why it cannot.


## What to report

Browser and version, operating system, and:

- every `FAIL` line,
- what §10 did (froze, or stopped cleanly),
- anything in §11 that misbehaved,
- anything red in the browser's developer console.

A `FAIL` on §2 is the serious one: it means this build of Diluvium
cannot catch errors in this browser, and the Lab should say so rather
than pretend.
